# Webcrawler BRACIS (2023, 2024 e 2025)

Este notebook implementa um webcrawler para o portal da SBC (SOL), focado nas edições do **BRACIS**.

## Objetivo

Coletar, para os anos **2023**, **2024** e **2025**, os metadados dos trabalhos publicados no BRACIS:

- Título
- Resumo
- Link do trabalho
- Ano e edição

Ao final, os dados são consolidados em um `DataFrame` do `pandas`.

## Como o crawler funciona

Fluxo geral:

1. Acessar o arquivo de edições do BRACIS no SOL.
2. Filtrar somente as edições dos anos alvo (`2023`, `2024`, `2025`).
3. Em cada edição, listar os links dos artigos.
4. Em cada artigo, extrair título e resumo.
5. Consolidar tudo em um único `DataFrame`.

O crawler inclui tratamento básico de erro de rede para não interromper toda a coleta caso um item falhe.

## Dependências e configuração

Bibliotecas utilizadas:

- `requests`: requisições HTTP
- `BeautifulSoup` (`bs4`): parse do HTML
- `pandas`: estrutura tabular e exportação

Parâmetros importantes:

- `ANOS_ALVO`: conjunto de anos desejados
- `SLEEP_SECONDS`: intervalo entre requisições para reduzir carga no servidor
- `REQUEST_TIMEOUT`: tempo limite por requisição

In [ ]:
import re
import time
from dataclasses import dataclass
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://sol.sbc.org.br/index.php"
BRACIS_SLUG = "bracis"
ANOS_ALVO = {2023, 2024, 2025}
REQUEST_TIMEOUT = 30
SLEEP_SECONDS = 0.2
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
)


## Implementação das funções

### Estruturas auxiliares

- `Edicao`: representa uma edição do evento com ano, título e link.

### Funções principais

- `baixar_soup`: faz download da página e converte para `BeautifulSoup`.
- `extrair_ano`: identifica o ano em um texto usando regex.
- `listar_edicoes_bracis`: coleta as edições do BRACIS e filtra os anos alvo.
- `listar_artigos_edicao`: obtém os artigos de uma edição.
- `extrair_titulo_resumo`: acessa a página do artigo e extrai título/resumo.
- `coletar_trabalhos_bracis`: orquestra todo o processo e retorna o `DataFrame` final.

### Estratégia de fallback para resumo

A coleta tenta, nessa ordem:

1. Bloco principal de resumo (`div.item.abstract`)
2. Meta tag `citation_abstract`
3. Meta tag `DC.Description`

In [ ]:
@dataclass(frozen=True)
class Edicao:
    ano: int
    titulo: str
    link: str


def baixar_soup(url: str, session: requests.Session) -> BeautifulSoup:
    resposta = session.get(url, timeout=REQUEST_TIMEOUT)
    resposta.raise_for_status()
    return BeautifulSoup(resposta.text, "html.parser")


def extrair_ano(texto: str) -> int | None:
    if not texto:
        return None
    match = re.search(r"\\b(20\\d{2})\\b", texto)
    return int(match.group(1)) if match else None


def listar_edicoes_bracis(
    session: requests.Session, anos_alvo: set[int]
) -> list[Edicao]:
    archive_url = f"{BASE_URL}/{BRACIS_SLUG}/issue/archive"
    soup = baixar_soup(archive_url, session)
    edicoes: list[Edicao] = []

    for bloco in soup.select("div.obj_issue_summary"):
        tag_titulo = bloco.select_one("a.title")
        if not tag_titulo:
            continue

        titulo_edicao = tag_titulo.get_text(" ", strip=True)
        ano_edicao = extrair_ano(titulo_edicao)

        if ano_edicao is None:
            tag_data = bloco.select_one("div.date_published span.value")
            ano_edicao = extrair_ano(tag_data.get_text(" ", strip=True) if tag_data else "")

        if ano_edicao not in anos_alvo:
            continue

        link_edicao = urljoin(archive_url, tag_titulo.get("href", "").strip())
        edicoes.append(Edicao(ano=ano_edicao, titulo=titulo_edicao, link=link_edicao))

    edicoes.sort(key=lambda item: (item.ano, item.titulo), reverse=True)
    return edicoes


def listar_artigos_edicao(url_edicao: str, session: requests.Session) -> list[dict[str, str]]:
    soup = baixar_soup(url_edicao, session)
    artigos: list[dict[str, str]] = []
    links_vistos: set[str] = set()

    for bloco in soup.select("div.obj_article_summary"):
        tag_artigo = bloco.select_one("div.title a")
        if not tag_artigo:
            continue

        link_artigo = urljoin(url_edicao, tag_artigo.get("href", "").strip())
        if not link_artigo or link_artigo in links_vistos:
            continue

        links_vistos.add(link_artigo)
        artigos.append(
            {
                "titulo_edicao": tag_artigo.get_text(" ", strip=True),
                "link_artigo": link_artigo,
            }
        )

    return artigos


def extrair_titulo_resumo(
    url_artigo: str, titulo_fallback: str, session: requests.Session
) -> tuple[str, str]:
    soup = baixar_soup(url_artigo, session)

    tag_titulo = soup.select_one("h1.page_title")
    titulo = tag_titulo.get_text(" ", strip=True) if tag_titulo else titulo_fallback

    if not titulo:
        tag_meta_titulo = soup.select_one('meta[name="citation_title"]')
        titulo = tag_meta_titulo.get("content", "").strip() if tag_meta_titulo else ""
    if not titulo:
        titulo = titulo_fallback

    resumo = ""
    tag_resumo = soup.select_one("div.item.abstract")
    if tag_resumo:
        tag_label = tag_resumo.select_one("h3.label")
        if tag_label:
            tag_label.decompose()
        resumo = tag_resumo.get_text(" ", strip=True)

    if not resumo:
        tag_meta_resumo = soup.select_one('meta[name="citation_abstract"]')
        if tag_meta_resumo:
            resumo = tag_meta_resumo.get("content", "").strip()

    if not resumo:
        for meta_desc in soup.select('meta[name="DC.Description"]'):
            texto = meta_desc.get("content", "").strip()
            if texto:
                resumo = texto
                break

    return titulo, resumo


def coletar_trabalhos_bracis(
    anos_alvo: set[int] = ANOS_ALVO, sleep_seconds: float = SLEEP_SECONDS
) -> pd.DataFrame:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    registros: list[dict[str, str | int]] = []
    edicoes = listar_edicoes_bracis(session, anos_alvo)

    print(f"Edições BRACIS encontradas para {sorted(anos_alvo)}: {len(edicoes)}")

    for edicao in edicoes:
        print(f"- {edicao.ano}: {edicao.titulo}")
        artigos = listar_artigos_edicao(edicao.link, session)
        print(f"  Artigos na edição: {len(artigos)}")

        for artigo in artigos:
            time.sleep(sleep_seconds)
            try:
                titulo, resumo = extrair_titulo_resumo(
                    artigo["link_artigo"], artigo["titulo_edicao"], session
                )
            except requests.RequestException as erro:
                print(f"  [aviso] erro em {artigo['link_artigo']}: {erro}")
                titulo = artigo["titulo_edicao"]
                resumo = ""

            registros.append(
                {
                    "Conferencia": "BRACIS",
                    "Ano": edicao.ano,
                    "Edicao": edicao.titulo,
                    "Titulo": titulo,
                    "Resumo": resumo,
                    "Link": artigo["link_artigo"],
                }
            )

    df = pd.DataFrame(
        registros,
        columns=["Conferencia", "Ano", "Edicao", "Titulo", "Resumo", "Link"],
    )

    if not df.empty:
        df = (
            df.drop_duplicates(subset=["Ano", "Titulo", "Link"])
            .sort_values(["Ano", "Titulo"], ascending=[False, True])
            .reset_index(drop=True)
        )

    return df


## Execução da coleta

A célula abaixo executa o crawler completo e monta o `DataFrame` consolidado em `df_bracis`.

Durante a execução, serão exibidas mensagens com:

- total de edições encontradas
- edição sendo processada
- quantidade de artigos por edição
- avisos pontuais de erro (se houver)

In [ ]:
df_bracis = coletar_trabalhos_bracis()
df_bracis.head(10)

## Inspeção rápida do resultado

Use esta célula para verificar:

- quantidade total de registros
- colunas disponíveis
- distribuição por ano

In [ ]:
print(f"Total de registros: {len(df_bracis)}")
print("\nColunas:", list(df_bracis.columns))
print("\nTrabalhos por ano:")
print(df_bracis["Ano"].value_counts().sort_index(ascending=False))

## Exportação para CSV

A célula seguinte salva os dados para uso posterior fora do notebook.

In [ ]:
arquivo_saida = "bracis_trabalhos_2023_2025.csv"
df_bracis.to_csv(arquivo_saida, index=False)
print(f"CSV salvo em: {arquivo_saida}")
print(f"Total de registros exportados: {len(df_bracis)}")

## Estrutura final do DataFrame

O `DataFrame` consolidado possui as colunas:

- `Conferencia`: nome do evento (fixo como BRACIS)
- `Ano`: ano da edição
- `Edicao`: título da edição
- `Titulo`: título do trabalho
- `Resumo`: resumo do trabalho
- `Link`: URL da página do trabalho

Esse formato está pronto para análises, filtros e visualizações.

## Observações

- Dependendo da sua conexão e da carga do servidor, a execução pode demorar alguns minutos.
- Se ocorrer falha de rede em um artigo específico, o crawler registra aviso e continua.
- Caso queira incluir novos anos, basta alterar o conjunto `ANOS_ALVO`.